# OLMo 2 Family — E1 Verbatim Trace Analysis (Stage-Aware, Compliant Cases)

**Project**: Corpus-Grounded Failure-Mode Taxonomy for Unsafe Compliance
**Notebook step**: v1 (numerical-only stage-aware analysis; span-text comparison is deferred to v2)

This notebook analyzes the E1 verbatim-trace results across **all eight OLMo 2 models**
(1B / 7B / 13B / 32B × base / instruct) on HarmBench Standard, considering the **three training
stages** that OLMo 2 actually went through:

| Stage      | Dataset             | Shared across models? |
|------------|---------------------|------------------------|
| pretrain   | `olmo-mix-1124`     | **Yes** — same for all OLMo 2 models |
| midtrain   | `dolmino-mix-1124`  | **Yes** — same for all OLMo 2 models |
| posttrain  | model-specific SFT→DPO→RLVR | **No** — only for instruct models, differs by size |

Because pretrain and midtrain are shared by every model, we treat them as a single
**`base_corpus`** unit (= pretrain ∪ midtrain). Post-training is then layered on top for
instruct models only.

**Three analysis units**:

- **`base_corpus`** = pretrain ∪ midtrain  → every OLMo 2 model
- **`posttrain`**   = model-specific post-training data → instruct only
- **`full`**        = base_corpus ∪ posttrain  → instruct only (for base models, `full` ≡ `base_corpus`)

**Union definition**: For each record, the union of two stages is computed by concatenating
their `all_maximal_spans` lists and re-applying OLMoTrace Algorithm 1 step 2 (non-maximal
suppression) on the same `response_ids`. This guarantees that LongestMatchLen, VerbatimCoverage,
and span-length distributions are exactly what you would get if you had built a single
combined index and run E1 once.

**Compliant cases only**: All metrics are computed over records with `hb_label == 1`
(the model complied with the unsafe HarmBench prompt). Non-compliant records are excluded
from E1 metric analysis but are used to compute ASR.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from collections import defaultdict

# Add parent directory to path for utils
from utils import build_row, union_spans, synth_record, covered_tokens, longest_run

# Model configurations — must match MODEL_CONFIGS in e1_verbatim_trace.py
MODELS = {
    # Base models  (no post_training/ folder on disk)
    'olmo2-1b':           {'out_dir': 'olmo2_1b',           'family': 'base',     'size': '1B',  'size_num': 1},
    'olmo2-7b':           {'out_dir': 'olmo2_7b',           'family': 'base',     'size': '7B',  'size_num': 7},
    'olmo2-13b':          {'out_dir': 'olmo2_13b',          'family': 'base',     'size': '13B', 'size_num': 13},
    'olmo2-32b':          {'out_dir': 'olmo2_32b',          'family': 'base',     'size': '32B', 'size_num': 32},
    # Instruct models  (have all three stages)
    'olmo2-1b-instruct':  {'out_dir': 'olmo2_1b_instruct',  'family': 'instruct', 'size': '1B',  'size_num': 1},
    'olmo2-7b-instruct':  {'out_dir': 'olmo2_7b_instruct',  'family': 'instruct', 'size': '7B',  'size_num': 7},
    'olmo2-13b-instruct': {'out_dir': 'olmo2_13b_instruct', 'family': 'instruct', 'size': '13B', 'size_num': 13},
    'olmo2-32b-instruct': {'out_dir': 'olmo2_32b_instruct', 'family': 'instruct', 'size': '32B', 'size_num': 32},
}

# Stage → subfolder mapping
STAGE_DIRS = {
    'pretrain':  'pretraining',
    'midtrain':  'mid_training',
    'posttrain': 'post_training',
}

# Plot style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': 'black',
    'axes.labelcolor': 'black',
    'text.color': 'black',
    'xtick.color': 'black',
    'ytick.color': 'black',
    'legend.facecolor': 'white',
    'legend.edgecolor': 'black',
    'figure.dpi': 120,
})

# Color scheme for stages
STAGE_COLORS = {
    'pretrain':    '#4a90d9',   # blue
    'midtrain':    '#7ab648',   # green
    'base_corpus': '#5a5a8a',   # purple-gray (combined)
    'posttrain':   '#e8854a',   # orange
    'full':        '#c1342b',   # red (everything)
}

## 0.2  Helper functions: union of stages

The union of two (or more) sets of maximal spans for the **same response** is computed by:

1. **Concatenating** the `all_maximal_spans` lists from each stage.
2. **Sorting** by `begin` ascending, then by `end` descending (so for tied starts, the longer
   span comes first).
3. **Suppressing non-maximal spans**: walk through the sorted list and keep a span only if its
   `end` strictly exceeds the running `max_end` seen so far. This is exactly OLMoTrace
   Algorithm 1 step 2, applied to the union.

After re-suppression, we re-derive the standard E1 metrics:

- `LongestMatchLen` = max span length
- `VerbatimCoverage` = |union of token positions covered| / `response_token_len`
- `num_maximal_spans`, `span_length_distribution.{min, max, mean, median}`

## 1.1  Data Loading & Summary

For each model we load up to three E1 result files (one per stage). For each compliant record:

- `pretrain`, `midtrain`, `posttrain` are loaded directly from disk (when available).
- `base_corpus` is computed by `union_spans([pretrain, midtrain], response_token_len)`.
- `full` is computed by `union_spans([pretrain, midtrain, posttrain], response_token_len)`
  for instruct models. For base models (no posttrain on disk), `full` is identical to
  `base_corpus` and is **not** stored as a separate row to avoid double-counting.

All rows are flattened into a single dataframe `df` with columns:

- key columns: `model`, `family`, `size`, `size_num`, `id`, `hb_label`, `compliance`, `prompt`,
  `response_token_len`, `repetition_ratio`
- per-stage columns: for each stage in `{pretrain, midtrain, base_corpus, posttrain, full}`,
  `LongestMatchLen_<stage>`, `VerbatimCoverage_<stage>`, `num_maximal_spans_<stage>`,
  `span_min_<stage>`, `span_max_<stage>`, `span_mean_<stage>`, `span_median_<stage>`.

In [3]:
# ---- Load per-stage records and compute synthetic stages per record ----
all_data = {}     # model_key -> {stage: {id: record}}
all_rows = []     # flat list, one row per (model, record_id)
hb_summary = []   # HarmBench label summary per model

ALL_STAGES = ['pretrain', 'midtrain', 'base_corpus', 'posttrain', 'full']

for model_key, cfg in MODELS.items():
    is_instruct = (cfg['family'] == 'instruct')

    # ---- 1) Read raw stage files ----
    stage_records = {}   # stage -> {record_id: record_dict}
    for stage_name, subdir in STAGE_DIRS.items():
        # Base models are not expected to have post_training/
        if stage_name == 'posttrain' and not is_instruct:
            continue

        filepath = os.path.join('..', 'results', cfg['out_dir'], subdir, 'e1_verbatim_standard.json')
        if not os.path.isfile(filepath):
            print(f"  [MISSING] {model_key} / {stage_name}: {filepath}")
            continue

        with open(filepath, 'r', encoding='utf-8') as f:
            records = json.load(f)
        stage_records[stage_name] = {r['id']: r for r in records}

    if not stage_records:
        print(f"  [SKIP] {model_key}: no stage files found.")
        continue

    # ---- 2) HarmBench label summary (read from any one stage; they share hb_label) ----
    any_stage = next(iter(stage_records.values()))
    hb_path = os.path.join('..', 'data', cfg['out_dir'], 'harmbench_standard_labeled.json')
    if os.path.isfile(hb_path):
        with open(hb_path, 'r', encoding='utf-8') as f:
            hb_records = json.load(f)
        n_total       = len(hb_records)
        n_compliant   = sum(1 for r in hb_records if r.get('hb_label') == 1)
        n_noncompliant = sum(1 for r in hb_records if r.get('hb_label') == 0)
    else:
        print(f"  [WARNING] {model_key}: labeled file not found at {hb_path}, using stage records only")
        n_total        = len(any_stage)
        n_compliant    = sum(1 for r in any_stage.values() if r.get('hb_label') == 1)
        n_noncompliant = sum(1 for r in any_stage.values() if r.get('hb_label') == 0)

    asr = n_compliant / n_total * 100 if n_total > 0 else 0.0
    hb_summary.append({
        'model': model_key,
        'family': cfg['family'],
        'size': cfg['size'],
        'total': n_total,
        'compliant': n_compliant,
        'non_compliant': n_noncompliant,
        'ASR': asr,
    })

    # ---- 3) Per-record processing: keep only compliant + successful E1 in ALL available stages ----
    # Use intersection of record ids across stages so unioning is well defined.
    common_ids = set.intersection(*[set(d.keys()) for d in stage_records.values()])

    valid_records = {}    # stage -> list of valid records (compliant + no error)
    for stage_name, recs_by_id in stage_records.items():
        valid_records[stage_name] = []

    valid_ids = []
    for rid in sorted(common_ids):
        ok = True
        for stage_name, recs_by_id in stage_records.items():
            r = recs_by_id[rid]
            if r.get('hb_label') != 1 or 'error' in r.get('e1', {}):
                ok = False
                break
        if ok:
            valid_ids.append(rid)

    all_data[model_key] = {stage: {} for stage in ALL_STAGES}

    # ---- 4) For each valid record, build per-stage rows including synthetic stages ----
    for rid in valid_ids:
        # Reference record (use pretrain copy for shared fields)
        ref_record = stage_records['pretrain'][rid] if 'pretrain' in stage_records else stage_records[next(iter(stage_records))][rid]
        L = ref_record['e1']['response_token_len']

        # Per-record raw stage records
        per_stage_rec = {}
        for stage_name in ['pretrain', 'midtrain', 'posttrain']:
            if stage_name in stage_records:
                per_stage_rec[stage_name] = stage_records[stage_name][rid]

        # Synthetic: base_corpus = pretrain ∪ midtrain
        if 'pretrain' in per_stage_rec and 'midtrain' in per_stage_rec:
            base_union = union_spans(
                [per_stage_rec['pretrain']['e1']['all_maximal_spans'],
                 per_stage_rec['midtrain']['e1']['all_maximal_spans']],
                L,
            )
            per_stage_rec['base_corpus'] = synth_record(ref_record, base_union)

        # Synthetic: full = base_corpus ∪ posttrain (instruct only)
        if is_instruct and 'base_corpus' in per_stage_rec and 'posttrain' in per_stage_rec:
            full_union = union_spans(
                [per_stage_rec['base_corpus']['e1']['all_maximal_spans'],
                 per_stage_rec['posttrain']['e1']['all_maximal_spans']],
                L,
            )
            per_stage_rec['full'] = synth_record(ref_record, full_union)

        # Save into all_data for downstream sections that need raw spans
        for stage_name, rec in per_stage_rec.items():
            all_data[model_key][stage_name][rid] = rec

        # ---- 5) Build a single flat row ----
        row = {
            'model':              model_key,
            'family':             cfg['family'],
            'size':               cfg['size'],
            'size_num':           cfg['size_num'],
            'id':                 rid,
            'hb_label':           ref_record.get('hb_label', -1),
            'compliance':         'Compliant',
            'prompt':             ref_record['prompt'],
            'response_token_len': L,
        }

        # repetition_ratio (stage-independent, computed from response text)
        words = ref_record['response'].split()
        n = 4
        if len(words) >= n:
            ngrams = [tuple(words[i:i+n]) for i in range(len(words) - n + 1)]
            row['repetition_ratio'] = 1.0 - len(set(ngrams)) / len(ngrams)
        else:
            row['repetition_ratio'] = 0.0

        # Per-stage E1 metric columns
        for stage_name in ALL_STAGES:
            if stage_name in per_stage_rec:
                e1 = per_stage_rec[stage_name]['e1']
                sd = e1['span_length_distribution']
                row[f'LongestMatchLen_{stage_name}']  = e1['LongestMatchLen']
                row[f'VerbatimCoverage_{stage_name}'] = e1['VerbatimCoverage']
                row[f'num_maximal_spans_{stage_name}'] = e1['num_maximal_spans']
                row[f'span_min_{stage_name}']     = sd['min']
                row[f'span_max_{stage_name}']     = sd['max']
                row[f'span_mean_{stage_name}']    = sd['mean']
                row[f'span_median_{stage_name}']  = sd['median']
            else:
                row[f'LongestMatchLen_{stage_name}']  = np.nan
                row[f'VerbatimCoverage_{stage_name}'] = np.nan
                row[f'num_maximal_spans_{stage_name}'] = np.nan
                row[f'span_min_{stage_name}']     = np.nan
                row[f'span_max_{stage_name}']     = np.nan
                row[f'span_mean_{stage_name}']    = np.nan
                row[f'span_median_{stage_name}']  = np.nan

        all_rows.append(row)

df = pd.DataFrame(all_rows)

# HarmBench evaluation summary
df_hb = pd.DataFrame(hb_summary)
order = [k for k in MODELS.keys() if k in all_data]
df_hb = df_hb.set_index('model').reindex(order)
df_hb['ASR'] = df_hb['ASR'].map('{:.1f}%'.format)

print("\n----- HarmBench Evaluation Summary -----")
print(df_hb[['family', 'size', 'total', 'compliant', 'non_compliant', 'ASR']].to_string())
print()

# E1 metrics summary (compliant cases only) — base_corpus and full are most informative
print(f"Total compliant records with E1: {len(df)}")
print(f"Models loaded: {len(all_data)} / {len(MODELS)}\n")

# Per-model summary on the most-comprehensive stage available for each model
def best_stage_col(row, metric):
    """For instruct: use full; for base: use base_corpus."""
    if row['family'] == 'instruct':
        return row[f'{metric}_full']
    else:
        return row[f'{metric}_base_corpus']

df['LML_best']   = df.apply(lambda r: best_stage_col(r, 'LongestMatchLen'),  axis=1)
df['Cov_best']   = df.apply(lambda r: best_stage_col(r, 'VerbatimCoverage'), axis=1)

summary = df.groupby('model').agg(
    N=('id', 'count'),
    family=('family', 'first'),
    size=('size', 'first'),
    LML_mean=('LML_best', 'mean'),
    LML_median=('LML_best', 'median'),
    LML_max=('LML_best', 'max'),
    Cov_mean=('Cov_best', 'mean'),
    resp_len_mean=('response_token_len', 'mean'),
    rep_ratio_mean=('repetition_ratio', 'mean'),
).round(2)

summary = summary.reindex(order)

print("----- Per-Model E1 Summary (Compliant Cases, best-available stage) -----")
print("  base models   → metrics on stage = base_corpus  (pretrain ∪ midtrain)")
print("  instruct models → metrics on stage = full       (pretrain ∪ midtrain ∪ posttrain)")
print()
print(summary.to_string())


----- HarmBench Evaluation Summary -----
                      family size  total  compliant  non_compliant    ASR
model                                                                    
olmo2-1b                base   1B    200         25            175  12.5%
olmo2-7b                base   7B    200         38            162  19.0%
olmo2-13b               base  13B    200         45            155  22.5%
olmo2-32b               base  32B    200         39            161  19.5%
olmo2-1b-instruct   instruct   1B    200         16            184   8.0%
olmo2-7b-instruct   instruct   7B    200          6            194   3.0%
olmo2-13b-instruct  instruct  13B    200          8            192   4.0%
olmo2-32b-instruct  instruct  32B    200         18            182   9.0%

Total compliant records with E1: 195
Models loaded: 8 / 8

----- Per-Model E1 Summary (Compliant Cases, best-available stage) -----
  base models   → metrics on stage = base_corpus  (pretrain ∪ midtrain)
  instruct mo

#### HarmBench Evaluation Summary
**Column definitions**
- `family`: model family — base (pretrain + midtrain only) or instruct (pretrain + midtrain + post-training으로 SFT→DPO→RLVR을 거친 모델)

- `size`: model parameter count (1B / 7B / 13B / 32B)
- `total`: number of HarmBench Standard prompts evaluated
- `compliant`: number of prompts where the model complied with the unsafe request (hb_label == 1 from the HarmBench classifier)
- `non_compliant`: number of prompts where the model refused or otherwise did not comply (hb_label == 0)
- `ASR (Attack Success Rate)`: compliant / total × 100% — fraction of unsafe prompts that successfully elicited a compliant response  
   전체 unsafe prompt 중에서 모델이 실제로 응한 비율 — HarmBench 공격이 "성공"한 비율

#### Per-Model E1 Summary
**Column definitions**
- `N`: number of compliant records used for E1 metric computation per model

- `LML_mean` (LongestMatchLen mean): mean of LongestMatchLen across compliant records, where per-record LongestMatchLen = the longest verbatim-matched span (in tokens) between the model response and the chosen training corpus  
compliant record 전체에 대한 LongestMatchLen의 평균. record 단위 LongestMatchLen은 모델 response와 선택된 학습 corpus 사이에서 verbatim으로 매칭되는 가장 긴 span의 토큰 길이
- `LML_median`: median of LongestMatchLen across compliant records  
compliant record 전체에 대한 LongestMatchLen의 중앙값. outlier에 덜 민감해서 분포의 "전형적인" verbatim trace 길이를 나타냄
- `LML_max`: maximum LongestMatchLen observed across all compliant records of this model  
해당 모델의 compliant record 중에서 관측된 LongestMatchLen의 최댓값. 가장 긴 verbatim 인용이 얼마나 길었는지
- `Cov_mean` (VerbatimCoverage mean): mean of VerbatimCoverage across compliant records, where per-record VerbatimCoverage = fraction of response tokens covered by any maximal matching span (L_min = 1)   
Record 단위 VerbatimCoverage는 response 토큰 중 maximal span 하나 이상에 의해 cover되는 토큰의 비율 (L_min = 1, 길이 제한 없음)
- `resp_len_mean`: mean response length in tokens across compliant records
- `rep_ratio_mean`: mean of repetition_ratio (word-level 4-gram seq-rep-4: 1 - |unique 4-grams| / |total 4-grams|) across compliant records; higher values indicate more repetitive / degenerate output  
compliant record의 repetition_ratio 평균. 단어 단위 4-gram의 seq-rep-4 정의: 1 - 고유 4-gram 수 / 전체 4-gram 수. 값이 높을수록 응답이 더 반복적/degenerate함

### 1.1.1  Sanity check: raw VerbatimCoverage per stage

Before running the L_min sweep in §1.3, we want to know whether `VerbatimCoverage` (with
L_min = 1, i.e. *any* verbatim match) is essentially saturated at 1.0 for one or more
stages. If it is, then the raw `VerbatimCoverage` column is uninformative and only the
`L_min ≥ 5` (and larger) sweep values carry signal.

For each model and each stage, this cell prints the min / mean / median / max of
`VerbatimCoverage`, plus the fraction of records with coverage exactly equal to 1.0.

In [4]:
rows = []
for model_key in order:
    sub = df[df['model'] == model_key]
    for stage in ALL_STAGES:
        col = f'VerbatimCoverage_{stage}'
        if col not in sub.columns:
            continue
        vals = sub[col].dropna()
        if len(vals) == 0:
            continue
        rows.append({
            'model':   model_key,
            'stage':   stage,
            'N':       len(vals),
            'min':     round(vals.min(),    4),
            'mean':    round(vals.mean(),   4),
            'median':  round(vals.median(), 4),
            'max':     round(vals.max(),    4),
            'frac=1.0': round((vals == 1.0).mean(), 3),
        })

df_cov_sanity = pd.DataFrame(rows)
print("----- Raw VerbatimCoverage (L_min=1) sanity check per (model, stage) -----")
print(df_cov_sanity.to_string(index=False))

----- Raw VerbatimCoverage (L_min=1) sanity check per (model, stage) -----
             model       stage  N  min   mean  median    max  frac=1.0
          olmo2-1b    pretrain 25  1.0 1.0000     1.0 1.0000     1.000
          olmo2-1b    midtrain 25  1.0 1.0000     1.0 1.0000     1.000
          olmo2-1b base_corpus 25  1.0 1.0000     1.0 1.0000     1.000
          olmo2-7b    pretrain 38  1.0 1.0000     1.0 1.0000     1.000
          olmo2-7b    midtrain 38  1.0 1.0000     1.0 1.0000     1.000
          olmo2-7b base_corpus 38  1.0 1.0000     1.0 1.0000     1.000
         olmo2-13b    pretrain 45  1.0 1.0000     1.0 1.0000     1.000
         olmo2-13b    midtrain 45  1.0 1.0000     1.0 1.0000     1.000
         olmo2-13b base_corpus 45  1.0 1.0000     1.0 1.0000     1.000
         olmo2-32b    pretrain 39  1.0 1.0000     1.0 1.0000     1.000
         olmo2-32b    midtrain 39  1.0 1.0000     1.0 1.0000     1.000
         olmo2-32b base_corpus 39  1.0 1.0005     1.0 1.0104     0.744
 o

In [7]:
# Inspect the first record where mismatch occurs
mk = 'olmo2-32b'
rid = 0

pre_e1 = all_data[mk]['pretrain'][rid]['e1']
mid_e1 = all_data[mk]['midtrain'][rid]['e1']

print(f"=== Record {rid} ===")
print(f"pretrain  response_token_len: {pre_e1['response_token_len']}")
print(f"midtrain  response_token_len: {mid_e1['response_token_len']}")
print()

# All maximal spans from each stage, sorted by begin
pre_spans = sorted(pre_e1['all_maximal_spans'], key=lambda s: s['begin'])
mid_spans = sorted(mid_e1['all_maximal_spans'], key=lambda s: s['begin'])

print(f"pretrain  num_maximal_spans: {len(pre_spans)}")
print(f"midtrain  num_maximal_spans: {len(mid_spans)}")
print()

print("First 5 pretrain spans:")
for s in pre_spans[:5]:
    print(f"  begin={s['begin']:4d}  end={s['end']:4d}  length={s['length']}")
print()
print("First 5 midtrain spans:")
for s in mid_spans[:5]:
    print(f"  begin={s['begin']:4d}  end={s['end']:4d}  length={s['length']}")
print()

# Check the largest span end values
print(f"pretrain  max end value: {max(s['end'] for s in pre_spans)}")
print(f"midtrain  max end value: {max(s['end'] for s in mid_spans)}")

=== Record 0 ===
pretrain  response_token_len: 558
midtrain  response_token_len: 559

pretrain  num_maximal_spans: 221
midtrain  num_maximal_spans: 231

First 5 pretrain spans:
  begin=   0  end=   4  length=4
  begin=   1  end=   6  length=5
  begin=   2  end=  13  length=11
  begin=   5  end=  14  length=9
  begin=   7  end=  15  length=8

First 5 midtrain spans:
  begin=   0  end=   2  length=2
  begin=   1  end=   5  length=4
  begin=   2  end=   6  length=4
  begin=   3  end=   9  length=6
  begin=   4  end=  10  length=6

pretrain  max end value: 558
midtrain  max end value: 559


In [8]:
mk = 'olmo2-32b'
rid = 0

pre_rec = all_data[mk]['pretrain'][rid]
mid_rec = all_data[mk]['midtrain'][rid]

print("=== Top-level keys ===")
print(f"pretrain: {list(pre_rec.keys())}")
print(f"midtrain: {list(mid_rec.keys())}")
print()
print("=== e1 keys ===")
print(f"pretrain e1: {list(pre_rec['e1'].keys())}")
print(f"midtrain e1: {list(mid_rec['e1'].keys())}")
print()

# Check if response_ids exists anywhere
for key in ['response_ids', 'input_ids', 'tokens']:
    if key in pre_rec:
        print(f"pretrain['{key}'][:10] = {pre_rec[key][:10]}")
    if key in pre_rec.get('e1', {}):
        print(f"pretrain['e1']['{key}'][:10] = {pre_rec['e1'][key][:10]}")
    if key in mid_rec:
        print(f"midtrain['{key}'][:10] = {mid_rec[key][:10]}")
    if key in mid_rec.get('e1', {}):
        print(f"midtrain['e1']['{key}'][:10] = {mid_rec['e1'][key][:10]}")

=== Top-level keys ===
pretrain: ['id', 'prompt', 'response', 'model', 'metadata', 'hb_label', 'e1']
midtrain: ['id', 'prompt', 'response', 'model', 'metadata', 'hb_label', 'e1']

=== e1 keys ===
pretrain e1: ['response_token_len', 'LongestMatchLen', 'VerbatimCoverage', 'num_maximal_spans', 'num_top_k_spans', 'span_length_distribution', 'all_maximal_spans', 'top_k_spans']
midtrain e1: ['response_token_len', 'LongestMatchLen', 'VerbatimCoverage', 'num_maximal_spans', 'num_top_k_spans', 'span_length_distribution', 'all_maximal_spans', 'top_k_spans', 'ExampleSnippets']



In [9]:
for mk in order:
    if 'midtrain' not in all_data[mk] or 'pretrain' not in all_data[mk]:
        continue
    n_diff = 0
    n_total = 0
    for rid in all_data[mk]['pretrain']:
        if rid not in all_data[mk]['midtrain']:
            continue
        n_total += 1
        L_pre = all_data[mk]['pretrain'][rid]['e1']['response_token_len']
        L_mid = all_data[mk]['midtrain'][rid]['e1']['response_token_len']
        if L_pre != L_mid:
            n_diff += 1
    print(f"  {mk:25s}: {n_diff:3d} / {n_total:3d} records have mismatched L (pretrain vs midtrain)")

  olmo2-1b                 :   0 /  25 records have mismatched L (pretrain vs midtrain)
  olmo2-7b                 :   0 /  38 records have mismatched L (pretrain vs midtrain)
  olmo2-13b                :   0 /  45 records have mismatched L (pretrain vs midtrain)
  olmo2-32b                :  10 /  39 records have mismatched L (pretrain vs midtrain)
  olmo2-1b-instruct        :   0 /  16 records have mismatched L (pretrain vs midtrain)
  olmo2-7b-instruct        :   0 /   6 records have mismatched L (pretrain vs midtrain)
  olmo2-13b-instruct       :   0 /   8 records have mismatched L (pretrain vs midtrain)
  olmo2-32b-instruct       :   0 /  18 records have mismatched L (pretrain vs midtrain)


## 1.2  LongestMatchLen (LML) Distribution — Cross-Model Comparison

`LongestMatchLen` (LML) = the longest verbatim-matched span in tokens between the model
response and the chosen training corpus, per record.

We show three panels, one for each analysis unit:

- **base_corpus** (pretrain ∪ midtrain) — applies to all 8 models
- **posttrain** — applies to instruct models only (4 models)
- **full** (base_corpus ∪ posttrain) — applies to instruct models only

For base models, only the `base_corpus` panel is populated.

In [ ]:
def boxplot_lml(stage, ax, title):
    data, labels, colors = [], [], []
    for model_key in order:
        col = f'LongestMatchLen_{stage}'
        sub = df[df['model'] == model_key]
        vals = sub[col].dropna().values
        if len(vals) == 0:
            continue
        data.append(vals)
        labels.append(model_key.replace('olmo2-', ''))
        colors.append('#4a90d9' if MODELS[model_key]['family'] == 'base' else '#e8854a')

    if not data:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title, fontsize=11)
        return

    bp = ax.boxplot(data, labels=labels, patch_artist=True,
                    showmeans=True, meanprops={'marker': 'D', 'markerfacecolor': 'black',
                                                'markeredgecolor': 'black', 'markersize': 5})
    for patch, c in zip(bp['boxes'], colors):
        patch.set_facecolor(c)
        patch.set_alpha(0.7)
    ax.set_ylabel('LongestMatchLen (tokens)', fontsize=10)
    ax.set_title(title, fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
boxplot_lml('base_corpus', axes[0], 'base_corpus  (pretrain ∪ midtrain)')
boxplot_lml('posttrain',   axes[1], 'posttrain  (instruct only)')
boxplot_lml('full',        axes[2], 'full  (base_corpus ∪ posttrain, instruct only)')
plt.suptitle('OLMo 2 Family: LongestMatchLen Distribution per Stage (Compliant Cases)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Numeric table: per-model LML mean/median/max for each stage
print("\n----- LongestMatchLen summary per (model, stage) -----")
rows = []
for model_key in order:
    sub = df[df['model'] == model_key]
    for stage in ['base_corpus', 'posttrain', 'full']:
        col = f'LongestMatchLen_{stage}'
        vals = sub[col].dropna()
        if len(vals) == 0:
            continue
        rows.append({
            'model':  model_key,
            'stage':  stage,
            'N':      len(vals),
            'mean':   round(vals.mean(),   2),
            'median': round(vals.median(), 1),
            'max':    int(vals.max()),
        })
print(pd.DataFrame(rows).to_string(index=False))

## 1.3  VerbatimCoverage with L_min Sweep

`VerbatimCoverage(L_min)` = fraction of response tokens covered by at least one maximal
span of length ≥ `L_min`.

In §1.1.1 we already saw the raw `VerbatimCoverage` (which is the L_min = 1 case). The
L_min sweep tells us how that coverage shrinks as we restrict attention to longer matches.

We sweep `L_min ∈ {1, 3, 5, 8, 10, 15, 20}` and compute the **per-model mean** coverage at
each threshold, separately for `base_corpus`, `posttrain`, and `full`. This is computed
**directly from `all_maximal_spans`** in `all_data`, so it works for both real and synthetic
stages.

In [ ]:
L_MIN_VALUES = [1, 3, 5, 8, 10, 15, 20]

def coverage_at_L(record, L_min):
    L = record['e1']['response_token_len']
    if L == 0:
        return 0.0
    covered = set()
    for s in record['e1']['all_maximal_spans']:
        if (s['end'] - s['begin']) >= L_min:
            for t in range(s['begin'], s['end']):
                covered.add(t)
    return len(covered) / L

def sweep_for_stage(stage):
    """Returns DataFrame: rows = models, columns = L_min values, values = mean coverage."""
    rows = {}
    for model_key in order:
        recs_by_id = all_data[model_key].get(stage, {})
        if not recs_by_id:
            continue
        per_lmin = []
        for L_min in L_MIN_VALUES:
            covs = [coverage_at_L(r, L_min) for r in recs_by_id.values()]
            per_lmin.append(np.mean(covs) if covs else np.nan)
        rows[model_key] = per_lmin
    return pd.DataFrame(rows, index=L_MIN_VALUES).T

sweep_base = sweep_for_stage('base_corpus')
sweep_post = sweep_for_stage('posttrain')
sweep_full = sweep_for_stage('full')

# Plot: 3 panels
fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)
for ax, sweep_df, title in zip(
    axes,
    [sweep_base, sweep_post, sweep_full],
    ['base_corpus  (pretrain ∪ midtrain)', 'posttrain  (instruct only)',
     'full  (base_corpus ∪ posttrain, instruct only)'],
):
    for model_key in sweep_df.index:
        color = '#4a90d9' if MODELS[model_key]['family'] == 'base' else '#e8854a'
        ls = '-' if MODELS[model_key]['family'] == 'instruct' else '--'
        ax.plot(L_MIN_VALUES, sweep_df.loc[model_key].values,
                marker='o', color=color, linestyle=ls, alpha=0.8,
                label=model_key.replace('olmo2-', ''))
    ax.set_xlabel('L_min (minimum span length, tokens)', fontsize=10)
    ax.set_title(title, fontsize=11)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc='upper right')
axes[0].set_ylabel('Mean VerbatimCoverage', fontsize=10)
plt.suptitle('OLMo 2 Family: VerbatimCoverage vs L_min (Compliant Cases)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("\n----- Mean VerbatimCoverage at each L_min  (stage = base_corpus) -----")
print(sweep_base.round(3).to_string())
print("\n----- Mean VerbatimCoverage at each L_min  (stage = posttrain, instruct only) -----")
print(sweep_post.round(3).to_string())
print("\n----- Mean VerbatimCoverage at each L_min  (stage = full, instruct only) -----")
print(sweep_full.round(3).to_string())

## 1.4  Span Length Distribution

Distribution of **all maximal span lengths** (not just the longest per record), aggregated
across all compliant records of each model. Shown as histograms.

Three rows of 4 panels each:
- Row 1: `base_corpus`  (all 8 models)
- Row 2: `posttrain`    (instruct only — base model panels are blank)
- Row 3: `full`         (instruct only)

In [ ]:
def collect_span_lengths(model_key, stage):
    recs_by_id = all_data[model_key].get(stage, {})
    lengths = []
    for r in recs_by_id.values():
        for s in r['e1']['all_maximal_spans']:
            lengths.append(s['end'] - s['begin'])
    return lengths

stages_to_plot = ['base_corpus', 'posttrain', 'full']
fig, axes = plt.subplots(3, 8, figsize=(28, 11), sharex=True)

for row_idx, stage in enumerate(stages_to_plot):
    for col_idx, model_key in enumerate(order):
        ax = axes[row_idx, col_idx]
        lengths = collect_span_lengths(model_key, stage)
        if not lengths:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes,
                    fontsize=10, color='gray')
            ax.set_title(f"{model_key.replace('olmo2-', '')}\n[{stage}]", fontsize=9)
            ax.set_xticks([])
            ax.set_yticks([])
            continue
        color = '#4a90d9' if MODELS[model_key]['family'] == 'base' else '#e8854a'
        ax.hist(lengths, bins=range(1, 41), color=color, alpha=0.7,
                edgecolor='white', linewidth=0.3)
        ax.set_yscale('log')
        ax.set_title(f"{model_key.replace('olmo2-', '')}\n[{stage}]  (n={len(lengths)})",
                     fontsize=9)
        if row_idx == 2:
            ax.set_xlabel('Span length (tokens)', fontsize=9)
        if col_idx == 0:
            ax.set_ylabel('Count (log)', fontsize=9)
        ax.grid(axis='y', alpha=0.3)

plt.suptitle('OLMo 2 Family: Maximal Span Length Distribution per Stage (Compliant Cases)',
             fontsize=13, y=1.005)
plt.tight_layout()
plt.show()

# Numeric per-model per-stage span length percentiles
print("\n----- Span length percentiles per (model, stage) -----")
rows = []
for model_key in order:
    for stage in stages_to_plot:
        lengths = collect_span_lengths(model_key, stage)
        if not lengths:
            continue
        arr = np.array(lengths)
        rows.append({
            'model':  model_key,
            'stage':  stage,
            'n_spans': len(arr),
            'p50':     int(np.percentile(arr, 50)),
            'p75':     int(np.percentile(arr, 75)),
            'p90':     int(np.percentile(arr, 90)),
            'p99':     int(np.percentile(arr, 99)),
            'max':     int(arr.max()),
        })
print(pd.DataFrame(rows).to_string(index=False))

## 1.5  Degenerate Response Detection

`repetition_ratio` (word-level 4-gram seq-rep-4) measures how often 4-grams repeat in the
response. It is computed directly from the response text and is therefore **stage-independent**
— the same value applies to all stages of a given record.

A response is flagged as *degenerate* when `repetition_ratio > 0.5`.

This metric exists primarily to filter out base-model outputs that loop on themselves
(e.g. "the the the the") and would otherwise inflate verbatim trace numbers spuriously.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10), sharey=False)
axes = axes.flatten()

for idx, model_key in enumerate(order):
    ax = axes[idx]
    sub = df[df['model'] == model_key]
    color = '#4a90d9' if MODELS[model_key]['family'] == 'base' else '#e8854a'
    ax.hist(sub['repetition_ratio'], bins=20, range=(0, 1),
            color=color, alpha=0.7, edgecolor='white', linewidth=0.3)
    label = model_key.replace('olmo2-', '')
    ax.set_title(f"{label} (N={len(sub)})", fontsize=10)
    ax.set_xlabel('repetition_ratio', fontsize=9)
    if idx % 4 == 0:
        ax.set_ylabel('Count', fontsize=9)
    ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='threshold=0.5')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('OLMo 2 Family: Repetition Ratio Distribution (Compliant Cases)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("\n--- Degenerate Detection (repetition_ratio > 0.5) ---")
for model_key in order:
    sub = df[df['model'] == model_key]
    n_degen = (sub['repetition_ratio'] > 0.5).sum()
    print(f"  {model_key:25s}: {n_degen:3d} / {len(sub):3d} "
          f"({n_degen/len(sub)*100:.1f}%) degenerate")

## 1.6  Base vs Instruct Comparison

Direct paired comparison between base and instruct at each model size. We use the
**most-comprehensive stage available for each model**:

- base    → `base_corpus`
- instruct → `full` (= `base_corpus` ∪ `posttrain`)

This gives an apples-to-apples comparison in the sense that both sides use *every corpus
the model was actually trained on*. The flip side is that the instruct side has access to
strictly more data (posttrain), so any instruct > base difference could be partly explained
by that extra data — §1.7 isolates how much of the instruct-side LML is explained by
posttrain alone.

In [ ]:
sizes = ['1B', '7B', '13B', '32B']
metrics = [
    ('LML_best', 'LongestMatchLen (best stage)'),
    ('Cov_best', 'VerbatimCoverage (best stage)'),
    ('repetition_ratio', 'repetition_ratio'),
]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, (col, ylabel) in zip(axes, metrics):
    base_means = []
    inst_means = []
    for s in sizes:
        b = df[(df['family'] == 'base')     & (df['size'] == s)][col].dropna()
        i = df[(df['family'] == 'instruct') & (df['size'] == s)][col].dropna()
        base_means.append(b.mean() if len(b) else np.nan)
        inst_means.append(i.mean() if len(i) else np.nan)

    x = np.arange(len(sizes))
    w = 0.35
    ax.bar(x - w/2, base_means, w, label='base',     color='#4a90d9', alpha=0.85)
    ax.bar(x + w/2, inst_means, w, label='instruct', color='#e8854a', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(sizes)
    ax.set_xlabel('Model size')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('OLMo 2: Base vs Instruct — best-available stage per side', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Also print a paired table
print("\n----- Paired base vs instruct (best stage per side) -----")
rows = []
for s in sizes:
    b = df[(df['family'] == 'base')     & (df['size'] == s)]
    i = df[(df['family'] == 'instruct') & (df['size'] == s)]
    rows.append({
        'size':         s,
        'base_N':       len(b),
        'inst_N':       len(i),
        'base_LML_mean':  round(b['LML_best'].mean(), 2)        if len(b) else np.nan,
        'inst_LML_mean':  round(i['LML_best'].mean(), 2)        if len(i) else np.nan,
        'base_Cov_mean':  round(b['Cov_best'].mean(), 4)        if len(b) else np.nan,
        'inst_Cov_mean':  round(i['Cov_best'].mean(), 4)        if len(i) else np.nan,
        'base_rep_mean':  round(b['repetition_ratio'].mean(),3) if len(b) else np.nan,
        'inst_rep_mean':  round(i['repetition_ratio'].mean(),3) if len(i) else np.nan,
    })
print(pd.DataFrame(rows).to_string(index=False))

## 1.6.5  Compliance vs Verbatim Trace

Instruction-tuned models have markedly **lower compliance rates (ASR)** on HarmBench than
their base counterparts — they are explicitly trained to refuse unsafe requests. But when
they *do* comply, do they exhibit *more* verbatim trace, the same amount, or less?

This scatter plot puts all 8 models on the (ASR, mean LML) plane to make the trade-off
visible at a glance. Each model contributes one point. The y-axis uses the
best-available stage (`full` for instruct, `base_corpus` for base) so the comparison is
fair to both families.

In [ ]:
# Build the scatter dataframe
rows = []
for model_key in order:
    sub = df[df['model'] == model_key]
    if len(sub) == 0:
        continue
    asr_str = df_hb.loc[model_key, 'ASR']         # e.g. "34.5%"
    asr = float(asr_str.rstrip('%'))
    rows.append({
        'model':     model_key,
        'family':    MODELS[model_key]['family'],
        'size':      MODELS[model_key]['size'],
        'size_num':  MODELS[model_key]['size_num'],
        'ASR':       asr,
        'N_compliant': len(sub),
        'mean_LML':  sub['LML_best'].mean(),
        'mean_Cov':  sub['Cov_best'].mean(),
        'mean_rep':  sub['repetition_ratio'].mean(),
    })
df_scatter = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Panel A: ASR vs mean LML
ax = axes[0]
for fam, color in [('base', '#4a90d9'), ('instruct', '#e8854a')]:
    sub = df_scatter[df_scatter['family'] == fam]
    ax.scatter(sub['ASR'], sub['mean_LML'],
               s=sub['size_num'] * 25 + 60,
               c=color, alpha=0.75, edgecolors='black', linewidths=1, label=fam)
    for _, r in sub.iterrows():
        ax.annotate(r['model'].replace('olmo2-', ''),
                    (r['ASR'], r['mean_LML']),
                    xytext=(6, 6), textcoords='offset points', fontsize=8)
ax.set_xlabel('ASR (% of HarmBench prompts complied with)')
ax.set_ylabel('Mean LongestMatchLen (best stage)')
ax.set_title('Compliance vs Verbatim Trace (LML)')
ax.legend()
ax.grid(alpha=0.3)

# Panel B: ASR vs mean VerbatimCoverage
ax = axes[1]
for fam, color in [('base', '#4a90d9'), ('instruct', '#e8854a')]:
    sub = df_scatter[df_scatter['family'] == fam]
    ax.scatter(sub['ASR'], sub['mean_Cov'],
               s=sub['size_num'] * 25 + 60,
               c=color, alpha=0.75, edgecolors='black', linewidths=1, label=fam)
    for _, r in sub.iterrows():
        ax.annotate(r['model'].replace('olmo2-', ''),
                    (r['ASR'], r['mean_Cov']),
                    xytext=(6, 6), textcoords='offset points', fontsize=8)
ax.set_xlabel('ASR (%)')
ax.set_ylabel('Mean VerbatimCoverage (best stage)')
ax.set_title('Compliance vs Verbatim Coverage')
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle('OLMo 2 Family: Compliance vs Verbatim Trace (size encoded as marker area)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("\n----- Compliance vs Verbatim Trace -----")
print(df_scatter[['model','family','size','ASR','N_compliant',
                  'mean_LML','mean_Cov','mean_rep']].round(3).to_string(index=False))

## 1.7  Post-training Stage Contribution Analysis  *(instruct models only)*

This is the central new analysis enabled by stage-aware decomposition. **Question:**
does the model's post-training corpus (SFT + DPO + RLVR data) contribute verbatim spans
to unsafe responses that are *not* already explainable by pretrain ∪ midtrain?

If yes, this is direct evidence that **instruction-tuning data itself helps form unsafe
responses**, not just that the base model knew the content.

### Definitions

For each compliant record from an instruct model, define on the same `response_ids`:

- `coverage_base(t)`   = 1 iff token `t` is covered by some maximal span in `base_corpus`
- `coverage_post(t)`   = 1 iff token `t` is covered by some maximal span in `posttrain`
- `coverage_full(t)`   = `coverage_base(t)  OR  coverage_post(t)`

Then the **post-train-unique** token set is
`{ t : coverage_post(t) == 1 AND coverage_base(t) == 0 }`.

### Metrics

- **(M1) `posttrain_unique_coverage`** (per record): `|posttrain-unique tokens| / response_token_len`
- **(M2) `posttrain_unique_LML`** (per record): the longest single contiguous run of
  posttrain-unique tokens. (This is *not* a maximal span in the OLMoTrace sense — it's the
  longest contiguous stretch of token positions that posttrain explains and base_corpus does
  not.)
- **(M3) Posttrain-unique span samples**: any maximal posttrain span that has at least one
  token *not* covered by `base_corpus`. Listed by record; we count them and (in v2) will
  also display their decoded text.
- **(M4) Stage-of-origin classification**: classify each record as
    - `pretrain_only`     — no midtrain or posttrain unique contribution
    - `midtrain_added`    — midtrain adds unique tokens beyond pretrain
    - `posttrain_added`   — posttrain adds unique tokens beyond `base_corpus` (this is the key category)
    - `posttrain_dominant` — `posttrain_unique_coverage` > `pretrain_only_coverage`

In [ ]:
def covered_tokens(spans, L):
    s = set()
    for sp in spans:
        for t in range(sp['begin'], sp['end']):
            s.add(t)
    return s

def longest_run(positions, L):
    """Longest contiguous run of integers in `positions` (a set), within [0, L)."""
    if not positions:
        return 0
    best = 0
    cur  = 0
    for t in range(L):
        if t in positions:
            cur += 1
            if cur > best:
                best = cur
        else:
            cur = 0
    return best

contrib_rows = []
unique_post_span_rows = []  # M3: catalog of per-record posttrain-unique span counts

instruct_models = [m for m in order if MODELS[m]['family'] == 'instruct']

for model_key in instruct_models:
    base_recs = all_data[model_key].get('base_corpus', {})
    post_recs = all_data[model_key].get('posttrain',   {})
    full_recs = all_data[model_key].get('full',        {})
    pre_recs  = all_data[model_key].get('pretrain',    {})
    mid_recs  = all_data[model_key].get('midtrain',    {})

    common_ids = set(base_recs.keys()) & set(post_recs.keys()) & set(full_recs.keys())

    for rid in common_ids:
        L = full_recs[rid]['e1']['response_token_len']

        cov_pre  = covered_tokens(pre_recs[rid]['e1']['all_maximal_spans'],  L)
        cov_mid  = covered_tokens(mid_recs[rid]['e1']['all_maximal_spans'],  L)
        cov_base = covered_tokens(base_recs[rid]['e1']['all_maximal_spans'], L)
        cov_post = covered_tokens(post_recs[rid]['e1']['all_maximal_spans'], L)

        post_unique = cov_post - cov_base
        mid_unique_over_pre = cov_mid - cov_pre   # for M4 classification

        # M1
        m1 = len(post_unique) / L if L > 0 else 0.0
        # M2
        m2 = longest_run(post_unique, L)
        # M3: count posttrain spans that contribute at least one unique token
        n_unique_spans = 0
        max_unique_span_len = 0
        for sp in post_recs[rid]['e1']['all_maximal_spans']:
            sp_tokens = set(range(sp['begin'], sp['end']))
            uniq = sp_tokens - cov_base
            if uniq:
                n_unique_spans += 1
                if len(uniq) > max_unique_span_len:
                    max_unique_span_len = len(uniq)

        # M4: stage-of-origin
        if len(post_unique) > 0:
            pretrain_only_cov = len(cov_pre - cov_mid - cov_post) / L if L > 0 else 0.0
            if m1 > pretrain_only_cov:
                stage_origin = 'posttrain_dominant'
            else:
                stage_origin = 'posttrain_added'
        elif len(mid_unique_over_pre) > 0:
            stage_origin = 'midtrain_added'
        else:
            stage_origin = 'pretrain_only'

        contrib_rows.append({
            'model':                       model_key,
            'size':                        MODELS[model_key]['size'],
            'id':                          rid,
            'response_token_len':          L,
            'cov_base':                    round(len(cov_base) / L, 4) if L > 0 else 0.0,
            'cov_post':                    round(len(cov_post) / L, 4) if L > 0 else 0.0,
            'posttrain_unique_coverage':   round(m1, 4),
            'posttrain_unique_LML':        m2,
            'n_posttrain_unique_spans':    n_unique_spans,
            'max_unique_span_len':         max_unique_span_len,
            'stage_origin':                stage_origin,
        })

df_contrib = pd.DataFrame(contrib_rows)
print(f"\n----- §1.7 contribution dataframe: {len(df_contrib)} rows -----\n")

# Per-model summary
print("----- Per-model M1 (posttrain_unique_coverage) and M2 (posttrain_unique_LML) -----")
agg = df_contrib.groupby('model').agg(
    N=('id', 'count'),
    M1_mean=('posttrain_unique_coverage', 'mean'),
    M1_max=('posttrain_unique_coverage',  'max'),
    M1_frac_pos=('posttrain_unique_coverage', lambda s: (s > 0).mean()),
    M2_mean=('posttrain_unique_LML', 'mean'),
    M2_max=('posttrain_unique_LML',  'max'),
    M2_ge8=('posttrain_unique_LML',  lambda s: (s >= 8).sum()),
    n_uniq_span_total=('n_posttrain_unique_spans', 'sum'),
    max_uniq_span_len=('max_unique_span_len', 'max'),
).round(4)
agg = agg.reindex([m for m in instruct_models if m in agg.index])
print(agg.to_string())

In [ ]:
# Plot: M1 distribution + M2 distribution per instruct model
fig, axes = plt.subplots(2, 4, figsize=(20, 9), sharey='row')

for col_idx, model_key in enumerate(instruct_models):
    sub = df_contrib[df_contrib['model'] == model_key]

    # Top row: M1 histogram
    ax = axes[0, col_idx]
    if len(sub):
        ax.hist(sub['posttrain_unique_coverage'], bins=20, range=(0, max(0.01, sub['posttrain_unique_coverage'].max() + 0.01)),
                color='#e8854a', alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.set_title(f"{model_key.replace('olmo2-', '')}\nM1: posttrain-unique coverage", fontsize=10)
    ax.set_xlabel('Fraction of response tokens', fontsize=9)
    if col_idx == 0:
        ax.set_ylabel('# records', fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    # Bottom row: M2 histogram
    ax = axes[1, col_idx]
    if len(sub):
        max_m2 = max(1, int(sub['posttrain_unique_LML'].max()))
        ax.hist(sub['posttrain_unique_LML'], bins=range(0, max_m2 + 2),
                color='#c1342b', alpha=0.8, edgecolor='white', linewidth=0.3)
        ax.axvline(x=8, color='black', linestyle='--', alpha=0.5, label='L=8')
        ax.legend(fontsize=8)
    ax.set_title(f"{model_key.replace('olmo2-', '')}\nM2: longest unique posttrain run", fontsize=10)
    ax.set_xlabel('Longest run (tokens)', fontsize=9)
    if col_idx == 0:
        ax.set_ylabel('# records', fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('§1.7  Post-training Stage Contribution (instruct models only)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# M4: stage-of-origin classification breakdown
print("----- M4: Stage-of-origin classification per instruct model -----")
m4 = df_contrib.groupby(['model', 'stage_origin']).size().unstack(fill_value=0)
m4 = m4.reindex(columns=['pretrain_only', 'midtrain_added', 'posttrain_added', 'posttrain_dominant'],
                fill_value=0)
m4 = m4.reindex([m for m in instruct_models if m in m4.index])
m4['total'] = m4.sum(axis=1)
print(m4.to_string())
print()
m4_pct = m4.iloc[:, :-1].div(m4['total'], axis=0).mul(100).round(1)
m4_pct.columns = [c + '_%' for c in m4_pct.columns]
print("----- Same, as percentages of each model's compliant records -----")
print(m4_pct.to_string())

## 1.8  Pretrain vs Midtrain Decomposition  *(bonus)*

Throughout §1.2–§1.7 we treated `base_corpus = pretrain ∪ midtrain` as a single unit.
Here we crack it open one more level to see how much each of the two shared corpora
actually contributes inside `base_corpus`.

We ask the symmetric questions:

- **`pretrain_unique_coverage`**: tokens covered by `pretrain` but **not** by `midtrain`.
- **`midtrain_unique_coverage`**: tokens covered by `midtrain` but **not** by `pretrain`.
- **`shared_coverage`**:           tokens covered by both.

Since pretrain and midtrain are shared across all OLMo 2 models, this analysis is run on
**all 8 models**, not just instruct.

In [ ]:
decomp_rows = []
for model_key in order:
    pre_recs = all_data[model_key].get('pretrain', {})
    mid_recs = all_data[model_key].get('midtrain', {})
    common_ids = set(pre_recs.keys()) & set(mid_recs.keys())

    for rid in common_ids:
        L = pre_recs[rid]['e1']['response_token_len']
        if L == 0:
            continue
        cov_pre = covered_tokens(pre_recs[rid]['e1']['all_maximal_spans'], L)
        cov_mid = covered_tokens(mid_recs[rid]['e1']['all_maximal_spans'], L)

        pre_only = cov_pre - cov_mid
        mid_only = cov_mid - cov_pre
        shared   = cov_pre & cov_mid

        decomp_rows.append({
            'model':                     model_key,
            'family':                    MODELS[model_key]['family'],
            'size':                      MODELS[model_key]['size'],
            'id':                        rid,
            'response_token_len':        L,
            'cov_pretrain':              round(len(cov_pre) / L, 4),
            'cov_midtrain':              round(len(cov_mid) / L, 4),
            'pretrain_unique_coverage':  round(len(pre_only) / L, 4),
            'midtrain_unique_coverage':  round(len(mid_only) / L, 4),
            'shared_coverage':           round(len(shared)   / L, 4),
        })

df_decomp = pd.DataFrame(decomp_rows)

print("----- §1.8 Pretrain vs Midtrain decomposition (means per model) -----")
agg = df_decomp.groupby('model').agg(
    N=('id', 'count'),
    cov_pretrain=('cov_pretrain', 'mean'),
    cov_midtrain=('cov_midtrain', 'mean'),
    pretrain_unique=('pretrain_unique_coverage', 'mean'),
    midtrain_unique=('midtrain_unique_coverage', 'mean'),
    shared=('shared_coverage', 'mean'),
).round(4)
agg = agg.reindex(order)
print(agg.to_string())

In [ ]:
# Stacked bar plot: per model, fraction of response tokens that are
#   pretrain_only / shared / midtrain_only / unmatched
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(order))
w = 0.7

# Compute means per model
pre_only = []
shared   = []
mid_only = []
unmatched = []
for model_key in order:
    sub = df_decomp[df_decomp['model'] == model_key]
    if len(sub) == 0:
        pre_only.append(0); shared.append(0); mid_only.append(0); unmatched.append(0)
        continue
    p = sub['pretrain_unique_coverage'].mean()
    s = sub['shared_coverage'].mean()
    m = sub['midtrain_unique_coverage'].mean()
    u = max(0.0, 1.0 - p - s - m)
    pre_only.append(p); shared.append(s); mid_only.append(m); unmatched.append(u)

ax.bar(x, pre_only,   w, label='pretrain only',  color='#4a90d9')
ax.bar(x, shared,     w, bottom=pre_only,        label='shared',         color='#5a5a8a')
bot2 = [p + s for p, s in zip(pre_only, shared)]
ax.bar(x, mid_only,   w, bottom=bot2,            label='midtrain only',  color='#7ab648')
bot3 = [b + m for b, m in zip(bot2, mid_only)]
ax.bar(x, unmatched,  w, bottom=bot3,            label='unmatched',      color='#cccccc')

ax.set_xticks(x)
ax.set_xticklabels([m.replace('olmo2-', '') for m in order], rotation=30, ha='right')
ax.set_ylabel('Mean fraction of response tokens')
ax.set_title('§1.8  Pretrain vs Midtrain decomposition (mean over compliant records)')
ax.set_ylim(0, 1.05)
ax.legend(loc='upper right', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

This notebook decomposes E1 verbatim trace by training stage for the full OLMo 2 family
(8 models, all available compliant records). Key sections:

- **§1.1** loads pretrain / midtrain / posttrain stage results, then synthesizes
  `base_corpus = pretrain ∪ midtrain` and `full = base_corpus ∪ posttrain` per record via
  span-list union + non-maximal suppression. Sanity check (§1.1.1) shows raw VerbatimCoverage
  per stage.
- **§1.2** shows LongestMatchLen distribution per stage (`base_corpus`, `posttrain`, `full`).
- **§1.3** sweeps L_min ∈ {1, 3, 5, 8, 10, 15, 20} for each stage to expose the part of
  coverage that is non-trivial (≥ 5 or ≥ 8 token spans).
- **§1.4** shows the maximal span-length distribution per (model, stage).
- **§1.5** detects degenerate responses via `repetition_ratio > 0.5` (stage-independent).
- **§1.6** pairs base vs instruct on the best-available stage per side.
- **§1.6.5** plots ASR vs verbatim trace to surface the "instruct refuses more, but when it
  complies, what does the trace look like?" trade-off.
- **§1.7** is the headline analysis: posttrain-unique contribution per instruct model
  (M1 coverage, M2 longest unique run, M3 unique-span counts, M4 stage-of-origin classification).
- **§1.8** decomposes `base_corpus` itself into pretrain-only / shared / midtrain-only.

Span-text inspection of post-training-unique spans (qualitative evidence for instruction-tuning
data driving unsafe response formation) is deferred to **v2** of this notebook.